In [ ]:
# =============================================================
# OpenQSim Phase 0A Benchmark Sweep — Kaggle T4 GPU
# =============================================================
# HOW TO USE:
#   1. Create a new Kaggle Notebook with GPU T4 x2 accelerator enabled.
#   2. Add your secrets in the Kaggle UI (Settings > Add-ons > Secrets):
#        KAGGLE_USERNAME  → your Kaggle username
#        KAGGLE_KEY       → your Kaggle API key
#        NVIDIA_API_KEY   → primary NVIDIA NIM key  (nvapi-...)
#        NVIDIA_API_KEY_1 → secondary NVIDIA NIM key (nvapi-...)
#        GROQ_API_KEY     → Groq key for advisor fallback (gsk_...)
#   3. Paste this entire cell into the notebook and run it.
#   4. The sweep writes ~840 JSON records, assembles a dataset, and
#      pushes it to your Kaggle Dataset.
# =============================================================

import os, subprocess, sys

# ── 1. Install dependencies ───────────────────────────────────────────────
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'qiskit>=2.0,<3.0',
    'qiskit-aer>=0.17',
    'pynvml',
    'nvidia-ml-py',
    'pyyaml',
    'pandas',
    'kaggle',
    'requests',
    'python-dotenv',
    'networkx',
])
print('[OK] Dependencies installed')

# ── 2. Clone the repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/lekkalaharsha/opensim-ai'  # <-- REPLACE if different
REPO_DIR = '/kaggle/working/OpenQSim'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    print(f'[OK] Cloned repo to {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
    print('[OK] Repo updated')

sys.path.insert(0, REPO_DIR)

# ── 3. Set environment variables from Kaggle Secrets ─────────────────────
from kaggle_secrets import UserSecretsClient  # available inside Kaggle notebooks
secrets = UserSecretsClient()

def _get_secret(name, required=True):
    try:
        val = secrets.get_secret(name)
        if val:
            os.environ[name] = val
            print(f'[OK] Secret {name} loaded')
            return val
    except Exception:
        pass
    if required:
        print(f'WARN Secret {name} not found — add it in Kaggle Secrets')
    return ''

_get_secret('KAGGLE_USERNAME')
_get_secret('KAGGLE_KEY')
_get_secret('NVIDIA_API_KEY')
_get_secret('NVIDIA_API_KEY_1', required=False)
_get_secret('GROQ_API_KEY', required=False)

# Signal the runner to use dual NVIDIA keys when both are present
if os.environ.get('NVIDIA_API_KEY_1'):
    os.environ['NVIDIA_API_KEY_COUNT'] = '2'
    print('[OK] Dual NVIDIA key mode enabled')

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
KAGGLE_DATASET  = f'{KAGGLE_USERNAME}/openqsim-benchmarks'
OUTPUT_DIR      = '/kaggle/working/benchmark_outputs'
SWEEP_CONFIG    = f'{REPO_DIR}/benchmark/sweep_config_0a.yaml'

print(f'[OK] Pushing to dataset: {KAGGLE_DATASET}')

# ── 4. Run the Phase 0A sweep ────────────────────────────────────────────
from kaggle.runner import KaggleRunner

runner = KaggleRunner(
    sweep_config_path=SWEEP_CONFIG,
    output_dir=OUTPUT_DIR,
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset=KAGGLE_DATASET,
    use_advisor=True,     # records LLM advisor predictions alongside timings
)

sweep_result = runner.run()

print(f'\nSweep done: {sweep_result.completed_count}/{sweep_result.total_combinations} records')
print(f'  OOM:    {sweep_result.oom_count}')
print(f'  Errors: {sweep_result.error_count}')
print(f'  Time:   {sweep_result.duration_seconds/3600:.1f}h')

# ── 5. Assemble the dataset ───────────────────────────────────────────────
import subprocess as sp
sp.check_call([
    sys.executable,
    f'{REPO_DIR}/scripts/run_sweep.py',
    '--assemble-only',
    '--output-dir', OUTPUT_DIR,
    '--no-push',
])

# ── 6. Final push to Kaggle Dataset ──────────────────────────────────────
import os, json
from pathlib import Path

dataset_dir = Path(OUTPUT_DIR)
metadata_path = dataset_dir / 'dataset-metadata.json'
metadata_path.write_text(json.dumps({
    'title': 'OpenQSim Benchmark Outputs',
    'id': KAGGLE_DATASET,
    'licenses': [{'name': 'MIT'}]
}))

import subprocess
result = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p', str(dataset_dir),
     '-m', f'Phase 0A sweep: {sweep_result.completed_count} records'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'[OK] Dataset pushed to {KAGGLE_DATASET}')
    print(f'     View at: https://www.kaggle.com/datasets/{KAGGLE_DATASET}')
else:
    print(f'WARN Push failed: {result.stderr[:300]}')
    print(f'     Data saved locally at {OUTPUT_DIR}')

print('\n=== Phase 0A complete ===')
